Requisitos:
- Uma cláusula que filtra dados usando ao menos dois operadores lógicos;  
- Uma função de agregação; 
- Uma função condicional; 
- Uma função de conversão; 
- Uma função de data;  
- Uma função de string 

In [9]:
import boto3
import pandas as pd
import io

In [10]:
# Acesso ao objeto no bucket

AWS_PROFILE = "localdesafio4"
session = boto3.Session(profile_name=AWS_PROFILE)
s3_client = session.client("s3")

BUCKET_NAME = "dados-renavam"
OBJETO_S3 = "dados_acidentes.csv"

response = s3_client.get_object(Bucket=BUCKET_NAME, Key=OBJETO_S3)

df = pd.read_csv(io.StringIO(response["Body"].read().decode("utf-8")))


In [11]:
# Óbitos por gênero e faixa de idade entre 2018 e 2024

df_1 = df[(df["qtde_obitos"] > 0) & (df["faixa_idade"] != "NA")]

q_1 = df_1.pivot_table(
    index="faixa_idade", 
    columns="genero", 
    values="qtde_obitos", 
    aggfunc="sum", 
    fill_value=0
).reset_index()

q_1.columns = ["faixa_idade", "feminino", "masculino"]

In [12]:
# Acidentes durante feriados ao passar dos anos

df["data_acidente"] = pd.to_datetime(df["data_acidente"])

feriados = {
    (1, 1): "Confraternização Universal",
    (4, 21): "Tiradentes",
    (5, 1): "Dia do Trabalhador",
    (9, 7): "Independência do Brasil",
    (10, 12): "Nossa Senhora Aparecida",
    (11, 2): "Finados",
    (12, 25): "Natal"
}
df["feriado"] = df["data_acidente"].apply(lambda x: feriados[(x.month, x.day)])

q_2 = df.groupby(["data_acidente", "feriado"])["uf_acidente"].count().reset_index()
q_2.rename(columns={"uf_acidente": "qtde_acidentes"}, inplace=True)

In [13]:
# Relação de acidentes por estado

q_3 = df.groupby("uf_acidente").agg(
    total_obitos=("qtde_obitos", "sum"),
    total_acidentes=("uf_acidente", "count")
).reset_index()

q_3["percentual_obitos"] = (q_3["total_obitos"] / q_3["total_acidentes"]) * 100
q_3["percentual_obitos"] = q_3["percentual_obitos"].astype(float).round(2)

q_3.rename(columns={"uf_acidente": "estado"}, inplace=True)

In [15]:
# Output das pesquisas

print("Óbitos por gênero e faixa de idade entre 2018 e 2024:\n\n", q_1.to_string(index=False))

print("\n\nAcidentes durante feriados ao passar dos anos:\n\n", q_2.to_string(index=False))

print("\n\nRelação de acidentes por estado:\n\n", q_3.to_string(index=False))

Óbitos por gênero e faixa de idade entre 2018 e 2024:

 faixa_idade  feminino  masculino
       1-17        25         76
      18-24        62        369
      25-29        46        230
      30-34        38        184
      35-39        43        192
      40-44        29        158
      45-49        25        109
      50-54        21        125
      55-59        13         76
      60-64        15         68
      65-69        15         55
      70-74        14         37
      75-79         8         17
        80+        14         28


Acidentes durante feriados ao passar dos anos:

 data_acidente                    feriado  qtde_acidentes
   2018-01-01 Confraternização Universal            2536
   2018-04-21                 Tiradentes            2748
   2018-05-01         Dia do Trabalhador            2159
   2018-09-07    Independência do Brasil            2896
   2018-10-12    Nossa Senhora Aparecida            2781
   2018-11-02                    Finados            2464